# Generate Sentinel-2 Samples (Earth Engine)

This notebook reproduces the **Land Cover.pdf** JavaScript script in Python using the Earth Engine API.
It creates Google Drive export tasks for 12-band Sentinel-2 L2A imagery at specific locations.

Requirements:
- `earthengine-api` installed
- An authenticated Earth Engine account

Notes:
- Exports are sent to your Google Drive in the folders defined below.
- You must manually start tasks in the Earth Engine Tasks tab (or call `task.start()`).


In [1]:
import ee

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

# Google Drive folder names
DRIVE_FOLDER_SPECTRAL = "S2_ML_Dataset_Spectral"
DRIVE_FOLDER_MASKS = "S2_ML_Dataset_Masks"

# New base locations (exclude existing sample names)
base_locations = [
    {"name": "Alexandria", "lon": 29.9187, "lat": 31.2001},
    {"name": "GizaPyramids", "lon": 31.1342, "lat": 29.9792},
    {"name": "Hurghada", "lon": 33.8216, "lat": 27.2579},
    {"name": "MarsaMatruh", "lon": 27.2373, "lat": 31.3543},
    {"name": "PortSaid", "lon": 32.2903, "lat": 31.2653},
    {"name": "Suez", "lon": 32.5498, "lat": 29.9668},
    {"name": "Damietta", "lon": 31.8144, "lat": 31.4175},
    {"name": "AswanDam", "lon": 32.8794, "lat": 23.9706},
    {"name": "Ismailia", "lon": 32.2723, "lat": 30.5965},
    {"name": "Minya", "lon": 30.7503, "lat": 28.0871},
    {"name": "BeniSuef", "lon": 31.0994, "lat": 29.0661},
    {"name": "LuxorEast", "lon": 32.6396, "lat": 25.6872},
]

# Generate many locations around each base point
# GRID_SIZE=4 gives 16 points per base -> 12 * 16 = 192 exports
GRID_SIZE = 4
STEP_METERS = 2000  # spacing between grid points

def expand_locations(bases, grid_size=4, step_m=2000):
    half = (grid_size - 1) / 2.0
    locations = []
    for base in bases:
        lat = base["lat"]
        lon = base["lon"]
        # Approx meters->degrees
        lat_deg = step_m / 111320.0
        lon_deg = step_m / (111320.0 * max(0.2, abs(__import__("math").cos(lat * 3.14159265 / 180.0))))
        for i in range(grid_size):
            for j in range(grid_size):
                dlat = (i - half) * lat_deg
                dlon = (j - half) * lon_deg
                name = f"{base['name']}_g{i}{j}"
                locations.append({"name": name, "lon": lon + dlon, "lat": lat + dlat})
    return locations

locations = expand_locations(base_locations, GRID_SIZE, STEP_METERS)

# Most recent summer to avoid clouds
start_date = "2025-05-01"
end_date = "2025-08-28"

def mask_clouds(img):
    scl = img.select("SCL")
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
    )
    return img.updateMask(mask)

target_bands = [
    "B1", "B2", "B3", "B4", "B5", "B6", "B7",
    "B8", "B8A", "B9", "B11", "B12",
]

def dynamic_world_mask(region, start_date, end_date):
    dw = (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .select("label")
    )
    # Mode label over time
    dw_label = dw.mode().clip(region)

    # Dynamic World classes:
    # 0 water, 1 trees, 2 grass, 3 flooded veg, 4 crops,
    # 5 shrub, 6 built, 7 bare, 8 snow/ice
    # Map to: 0 Unknown, 1 Greenery, 2 Sand, 3 Water, 4 Cement
    from_vals = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    to_vals   = [3, 1, 1, 1, 1, 1, 4, 2, 0]
    mapped = dw_label.remap(from_vals, to_vals).rename("mask")
    return mapped

tasks = []
for loc in locations:
    region = ee.Geometry.Point([loc["lon"], loc["lat"]]).buffer(1500).bounds()

    s2_sr = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    )

    clean_col = s2_sr.map(mask_clouds)
    clean_l2a = clean_col.select(target_bands).median().clip(region)

    spectral_task = ee.batch.Export.image.toDrive(
        image=clean_l2a,
        description=f"{loc['name']}_Spectral_300px",
        folder=DRIVE_FOLDER_SPECTRAL,
        scale=10,
        region=region,
        maxPixels=1e8,
        fileFormat="GeoTIFF",
    )
    tasks.append(spectral_task)

    mask_img = dynamic_world_mask(region, start_date, end_date)
    mask_task = ee.batch.Export.image.toDrive(
        image=mask_img,
        description=f"{loc['name']}_Mask_300px",
        folder=DRIVE_FOLDER_MASKS,
        scale=10,
        region=region,
        maxPixels=1e8,
        fileFormat="GeoTIFF",
    )
    tasks.append(mask_task)

    print("Tasks created for:", loc["name"])

print("Check the Earth Engine Tasks tab to start exports.")



Successfully saved authorization token.
Tasks created for: Alexandria_g00
Tasks created for: Alexandria_g01
Tasks created for: Alexandria_g02
Tasks created for: Alexandria_g03
Tasks created for: Alexandria_g10
Tasks created for: Alexandria_g11
Tasks created for: Alexandria_g12
Tasks created for: Alexandria_g13
Tasks created for: Alexandria_g20
Tasks created for: Alexandria_g21
Tasks created for: Alexandria_g22
Tasks created for: Alexandria_g23
Tasks created for: Alexandria_g30
Tasks created for: Alexandria_g31
Tasks created for: Alexandria_g32
Tasks created for: Alexandria_g33
Tasks created for: GizaPyramids_g00
Tasks created for: GizaPyramids_g01
Tasks created for: GizaPyramids_g02
Tasks created for: GizaPyramids_g03
Tasks created for: GizaPyramids_g10
Tasks created for: GizaPyramids_g11
Tasks created for: GizaPyramids_g12
Tasks created for: GizaPyramids_g13
Tasks created for: GizaPyramids_g20
Tasks created for: GizaPyramids_g21
Tasks created for: GizaPyramids_g22
Tasks created for: 

## Move Exports Into This Repo

After exports finish, download the GeoTIFFs from Google Drive and place them here:
- `data/samples/imgs` for `*_Spectral_300px.tif`
- `data/samples/masks` for `*_Mask_300px.tif`

Example (after download):
```bash
mv ~/Downloads/*_Spectral_300px.tif /home/mohamed-ashraf/Desktop/projects/sat-project/data/samples/imgs/
mv ~/Downloads/*_Mask_300px.tif /home/mohamed-ashraf/Desktop/projects/sat-project/data/samples/masks/
```


## Optional: Start All Tasks Automatically

If you want to start them immediately from the notebook, run:

```python
for task in tasks:
    task.start()
```


In [2]:
for task in tasks:
    task.start()